# Immune integration: RNA-only training (large model)

Train `AmbientRegularizedSCVI` on 7-dataset immune integration (bone marrow + PBMC).
Uses early stopping with stratified validation split.

**Model**: n_hidden=1024, n_latent=256 (2x base model)

**Input**: `results/immune_integration/adata_rna.h5ad` + `scrublet_results.csv`
**Output**: Trained model + latent representation + UMAP

In [ ]:
import sys
import os

sys.path.insert(0, "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/docs/notebooks/immune_integration")

import scanpy as sc
import numpy as np
import pandas as pd
from matplotlib import rcParams
import scipy
import torch
import scvi
import regularizedvi

scvi.settings.progress_bar_style = "tqdm"
torch.set_float32_matmul_precision("high")
rcParams["pdf.fonttype"] = 42

In [ ]:
# Papermill parameters
results_folder = "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/immune_integration_rna_v2/"
marker_genes_csv = "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/docs/notebooks/known_marker_genes.csv"
filtering_option = "adaptive"
exclude_datasets = "tea_seq"
decoder_type_rna = "expected_RNA"
burst_size_intercept = 1.0
skip_training = 0
n_hidden = 1024
n_latent = 256
additive_bg_prior_alpha = 1.0
additive_bg_prior_beta = 100.0
use_additive_background = 1
regularise_background = 0
compute_pearson = 1
use_feature_scaling = 1
feature_scaling_prior_alpha = 200.0
feature_scaling_prior_beta = 200.0
library_log_means_centering_sensitivity = 1.0
library_log_vars_weight = 0.5
init_decoder_bias = "mean"
bg_init_gene_fraction = 0.2
decoder_bias_multiplier = None
residual_library_encoder = 1
library_obs_w_prior_rate = 1.0
# Items 4 + 7: unified dispersion hyper-prior + new knobs (papermill-exposed)
# dispersion_hyper_prior_mean / _alpha = None -> DECODER_TYPE_DEFAULTS.
# Replaces legacy dispersion_hyper_prior_beta + regularise_dispersion_prior (Item 4).
dispersion_hyper_prior_mean = None
dispersion_hyper_prior_alpha = None
dispersion_prior_direction = "inverse_sqrt"
dispersion_init = "prior"
dispersion_init_bio_frac = 0.99
stratify_validation_key = "harmonized_annotation+batch"
early_stopping_min_delta_per_feature = 0.0002
early_stopping_patience = 10
max_epochs = 2000
wandb_project = "regularizedVI"
wandb_name = "immune_rna_v2"
wandb_entity = None
wandb_notes = "RNA: 7-dataset immune integration v2"
wandb_group = "immune_integration"
z_sparsity_prior = None
n_active_latent_per_cell = 20
decoder_hidden_l1 = 0.0
hidden_activation_sparsity = 0
n_active_hidden_per_cell = 40
n_epochs_kl_warmup = 400
use_kl_z = 1
horseshoe_latent_z_prior_type = None
horseshoe_latent_z_encoder_fraction = 1.0
horseshoe_posterior_init_loc = 1.0
horseshoe_posterior_init_scale = 0.1
# Item 7: encoder / decoder knobs
var_init_scale = 0.1
use_softplus_var_activation = True
use_ard_z_sigma_scale = False
decoder_weight_l2 = 0
batch_size = 1024

## Data loading

In [ ]:
from data_loading_utils import load_atac_qc_metrics, load_scrublet_scores

data_folder = "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/immune_integration/"
rna_path = os.path.join(data_folder, "adata_rna.h5ad")
scrublet_path = os.path.join(data_folder, "scrublet_results.csv")
atac_qc_path = os.path.join(data_folder, "atac_qc_metrics.csv")

adata = sc.read_h5ad(rna_path)
print(f"Loaded: {adata.shape}")

# Exclude datasets if specified
if exclude_datasets is not None:
    _excl = [s.strip() for s in exclude_datasets.split(",")]
    n_before = adata.shape[0]
    adata = adata[~adata.obs["dataset"].isin(_excl)].copy()
    print(f"Excluded datasets {_excl}: {n_before} -> {adata.shape[0]} cells")

# Load scrublet and ATAC QC using safe reindex utilities
adata = load_scrublet_scores(adata, scrublet_path)
adata = load_atac_qc_metrics(adata, atac_qc_path)

# Ensure counts in X
if "counts" in adata.layers:
    adata.X = adata.layers["counts"]

print(f"\nDatasets: {adata.obs['dataset'].nunique()}")
print(f"Batches: {adata.obs['batch'].nunique()}")
print(f"Donors: {adata.obs['donor'].nunique()}")

## QC summary

In [ ]:
from regularizedvi.utils import print_qc_summary

print_qc_summary(adata)

In [ ]:
from regularizedvi.utils import plot_qc_histograms

plot_qc_histograms(adata)

## Cell filtering

In [ ]:
if filtering_option == "adaptive":
    from data_loading_utils import adaptive_qc_filter

    adata = adaptive_qc_filter(
        adata,
        min_counts=900,
        min_genes=600,
        min_fragments=1500,
        per_dataset_quantile=0.20,
    )
elif filtering_option == "compound":
    from regularizedvi.utils import compound_qc_filter

    global_cutoffs = {
        "total_counts": (900, 80000),
        "n_genes": (600, 10000),
        "total_fragments": (1500, 100000),
        "mt_frac": (None, 0.15),
        "doublet_score": (None, 0.20),
    }
    per_study_cutoffs = {
        "bone_marrow": {
            "total_counts": (800, 80000),
            "n_genes": (600, 10000),
            "total_fragments": (2000, 100000),
            "mt_frac": (None, 0.10),
            "doublet_score": (None, 0.19),
        },
        "covid_pbmc": {
            "total_counts": (1500, 80000),
            "n_genes": (900, 10000),
            "total_fragments": (2000, 100000),
            "mt_frac": (None, 0.10),
            "doublet_score": (None, 0.18),
        },
        "crohns_pbmc": {
            "total_counts": (1500, 80000),
            "n_genes": (900, 10000),
            "total_fragments": (2000, 100000),
            "mt_frac": (None, 0.10),
            "doublet_score": (None, 0.18),
        },
        "infant_adult_spleen": {
            "total_counts": (1500, 80000),
            "n_genes": (900, 10000),
            "total_fragments": (2000, 100000),
            "mt_frac": (None, 0.10),
            "doublet_score": (None, 0.18),
        },
        "lung_spleen_gse319044": {
            "total_counts": (1200, 80000),
            "n_genes": (900, 10000),
            "total_fragments": (1500, 100000),
            "mt_frac": (None, 0.10),
            "doublet_score": (None, 0.18),
        },
        "neat_seq_cd4t": {
            "total_counts": (1200, 80000),
            "n_genes": (900, 10000),
            "total_fragments": (1500, 100000),
            "mt_frac": (None, 0.10),
            "doublet_score": (None, 0.25),
        },
        "pbmc_tea_seq": {
            "total_counts": (1200, 80000),
            "n_genes": (600, 10000),
            "total_fragments": (1500, 100000),
            "mt_frac": (None, 0.25),
            "doublet_score": (None, 0.18),
        },
    }
    adata = compound_qc_filter(
        adata,
        per_study_cutoffs=per_study_cutoffs,
        global_cutoffs=global_cutoffs,
    )
else:
    raise ValueError(f"Unknown filtering_option: {filtering_option!r}. Use 'adaptive' or 'compound'.")

## Gene filtering

In [ ]:
from regularizedvi.utils import filter_genes

selected = filter_genes(
    adata,
    cell_count_cutoff=15,
    cell_percentage_cutoff2=0.01,
    nonz_mean_cutoff=1.04,
)
adata = adata[:, selected].copy()
print(f"Selected {adata.n_vars:,} genes")

In [ ]:
adata.layers["counts"] = adata.X

In [ ]:
from regularizedvi.utils import (
    coerce_papermill_params,
    finish_wandb,
    log_figure_to_wandb,
    setup_wandb_logger,
    wandb_config_from_locals,
)

params = coerce_papermill_params(
    skip_training=(skip_training, bool),
    exclude_datasets=(exclude_datasets, "str_or_none"),
    decoder_type_rna=(decoder_type_rna, "str_or_none"),
    burst_size_intercept=(burst_size_intercept, float),
    additive_bg_prior_alpha=(additive_bg_prior_alpha, float),
    additive_bg_prior_beta=(additive_bg_prior_beta, float),
    use_additive_background=(use_additive_background, bool),
    regularise_background=(regularise_background, bool),
    compute_pearson=(compute_pearson, bool),
    use_feature_scaling=(use_feature_scaling, bool),
    feature_scaling_prior_alpha=(feature_scaling_prior_alpha, float),
    feature_scaling_prior_beta=(feature_scaling_prior_beta, float),
    library_log_means_centering_sensitivity=(library_log_means_centering_sensitivity, "float_or_none"),
    library_log_vars_weight=(library_log_vars_weight, float),
    decoder_weight_l2=(decoder_weight_l2, float),
    init_decoder_bias=(init_decoder_bias, "str_or_none"),
    bg_init_gene_fraction=(bg_init_gene_fraction, "float_or_none"),
    decoder_bias_multiplier=(decoder_bias_multiplier, "float_or_none"),
    residual_library_encoder=(residual_library_encoder, bool),
    library_obs_w_prior_rate=(library_obs_w_prior_rate, float),
    dispersion_hyper_prior_mean=(dispersion_hyper_prior_mean, "float_or_none"),
    dispersion_hyper_prior_alpha=(dispersion_hyper_prior_alpha, "float_or_none"),
    dispersion_prior_direction=(dispersion_prior_direction, "str_or_none"),
    dispersion_init=(dispersion_init, "str_or_none"),
    dispersion_init_bio_frac=(dispersion_init_bio_frac, float),
    n_hidden=(n_hidden, int),
    n_latent=(n_latent, int),
    stratify_validation_key=(stratify_validation_key, "str_or_none"),
    early_stopping_min_delta_per_feature=(early_stopping_min_delta_per_feature, "float_or_none"),
    early_stopping_patience=(early_stopping_patience, int),
    max_epochs=(max_epochs, int),
    wandb_project=(wandb_project, "str_or_none"),
    wandb_name=(wandb_name, "str_or_none"),
    wandb_entity=(wandb_entity, "str_or_none"),
    wandb_notes=(wandb_notes, "str_or_none"),
    wandb_group=(wandb_group, "str_or_none"),
    z_sparsity_prior=(z_sparsity_prior, "str_or_none"),
    n_active_latent_per_cell=(n_active_latent_per_cell, float),
    decoder_hidden_l1=(decoder_hidden_l1, float),
    hidden_activation_sparsity=(hidden_activation_sparsity, bool),
    n_active_hidden_per_cell=(n_active_hidden_per_cell, float),
    n_epochs_kl_warmup=(n_epochs_kl_warmup, int),
    use_kl_z=(use_kl_z, bool),
    horseshoe_latent_z_prior_type=(horseshoe_latent_z_prior_type, "str_or_none"),
    horseshoe_latent_z_encoder_fraction=(horseshoe_latent_z_encoder_fraction, float),
    horseshoe_posterior_init_loc=(horseshoe_posterior_init_loc, float),
    horseshoe_posterior_init_scale=(horseshoe_posterior_init_scale, float),
    var_init_scale=(var_init_scale, "float_or_none"),
    use_softplus_var_activation=(use_softplus_var_activation, bool),
    use_ard_z_sigma_scale=(use_ard_z_sigma_scale, bool),
    batch_size=(batch_size, int),
)
skip_training = params["skip_training"]
exclude_datasets = params["exclude_datasets"]
decoder_type_rna = params["decoder_type_rna"]
burst_size_intercept = params["burst_size_intercept"]
additive_bg_prior_alpha = params["additive_bg_prior_alpha"]
additive_bg_prior_beta = params["additive_bg_prior_beta"]
use_additive_background = params["use_additive_background"]
regularise_background = params["regularise_background"]
compute_pearson = params["compute_pearson"]
use_feature_scaling = params["use_feature_scaling"]
feature_scaling_prior_alpha = params["feature_scaling_prior_alpha"]
feature_scaling_prior_beta = params["feature_scaling_prior_beta"]
library_log_means_centering_sensitivity = params["library_log_means_centering_sensitivity"]
library_log_vars_weight = params["library_log_vars_weight"]
decoder_weight_l2 = params["decoder_weight_l2"]
init_decoder_bias = params["init_decoder_bias"]
bg_init_gene_fraction = params["bg_init_gene_fraction"]
decoder_bias_multiplier = params["decoder_bias_multiplier"]
residual_library_encoder = params["residual_library_encoder"]
library_obs_w_prior_rate = params["library_obs_w_prior_rate"]
dispersion_hyper_prior_mean = params["dispersion_hyper_prior_mean"]
dispersion_hyper_prior_alpha = params["dispersion_hyper_prior_alpha"]
dispersion_prior_direction = params["dispersion_prior_direction"]
dispersion_init = params["dispersion_init"]
dispersion_init_bio_frac = params["dispersion_init_bio_frac"]
n_hidden = params["n_hidden"]
n_latent = params["n_latent"]
stratify_validation_key = params["stratify_validation_key"]
early_stopping_min_delta_per_feature = params["early_stopping_min_delta_per_feature"]
early_stopping_patience = params["early_stopping_patience"]
max_epochs = params["max_epochs"]
wandb_project = params["wandb_project"]
wandb_name = params["wandb_name"]
wandb_entity = params["wandb_entity"]
wandb_notes = params["wandb_notes"]
wandb_group = params["wandb_group"]
z_sparsity_prior = params["z_sparsity_prior"]
n_active_latent_per_cell = params["n_active_latent_per_cell"]
decoder_hidden_l1 = params["decoder_hidden_l1"]
hidden_activation_sparsity = params["hidden_activation_sparsity"]
n_active_hidden_per_cell = params["n_active_hidden_per_cell"]
n_epochs_kl_warmup = params["n_epochs_kl_warmup"]
use_kl_z = params["use_kl_z"]
horseshoe_latent_z_prior_type = params["horseshoe_latent_z_prior_type"]
horseshoe_latent_z_encoder_fraction = params["horseshoe_latent_z_encoder_fraction"]
horseshoe_posterior_init_loc = params["horseshoe_posterior_init_loc"]
horseshoe_posterior_init_scale = params["horseshoe_posterior_init_scale"]
var_init_scale = params["var_init_scale"]
use_softplus_var_activation = params["use_softplus_var_activation"]
use_ard_z_sigma_scale = params["use_ard_z_sigma_scale"]
batch_size = params["batch_size"]

decoder_type_dict = {"rna": decoder_type_rna}
n_hidden_dict = {"rna": n_hidden}
n_latent_dict = {"rna": n_latent}

# Item 4: single unified dispersion hyper-prior; None -> DECODER_TYPE_DEFAULTS.
# RegularizedMultimodalVI accepts scalar mean/alpha/direction and broadcasts across modalities.
print(
    f"dispersion hyper-prior: mean={dispersion_hyper_prior_mean}, "
    f"alpha={dispersion_hyper_prior_alpha}, direction={dispersion_prior_direction}"
)

os.makedirs(results_folder, exist_ok=True)

wandb_loggers, wandb_run = setup_wandb_logger(
    wandb_project=wandb_project,
    wandb_name=wandb_name,
    wandb_entity=wandb_entity,
    wandb_notes=wandb_notes,
    wandb_group=wandb_group,
    config=wandb_config_from_locals(locals(), skip=("params",)),
    results_folder=results_folder,
)

## Model setup and training

In [ ]:
import mudata

adata_for_mdata = adata.copy()
adata_for_mdata.X = adata_for_mdata.layers["counts"]  # setup_mudata reads .X
mdata = mudata.MuData({"rna": adata_for_mdata})
del adata_for_mdata  # free RAM

In [ ]:
regularizedvi.RegularizedMultimodalVI.setup_mudata(
    mdata,
    ambient_covariate_keys=["batch"],
    nn_conditioning_covariate_keys=["dataset", "donor"],
    feature_scaling_covariate_keys=["dataset", "donor"] if use_feature_scaling else [],
    dispersion_key="batch",
    library_size_key="batch",
    encoder_covariate_keys=False,
)

In [ ]:
model = regularizedvi.RegularizedMultimodalVI(
    mdata,
    n_hidden=n_hidden_dict,
    n_latent=n_latent_dict,
    n_layers=1,
    additive_background_modalities=["rna"],
    feature_scaling_modalities=["rna"] if use_feature_scaling else [],
    dispersion="gene-batch",
    decoder_type=decoder_type_dict,
    burst_size_intercept=burst_size_intercept,
    regularise_dispersion=True,
    dispersion_hyper_prior_mean=dispersion_hyper_prior_mean,
    dispersion_hyper_prior_alpha=dispersion_hyper_prior_alpha,
    dispersion_prior_direction=dispersion_prior_direction,
    dispersion_init=dispersion_init,
    dispersion_init_bio_frac=dispersion_init_bio_frac,
    regularise_background=regularise_background,
    library_log_vars_weight=library_log_vars_weight,
    library_log_means_centering_sensitivity=library_log_means_centering_sensitivity,
    feature_scaling_prior_alpha=feature_scaling_prior_alpha,
    feature_scaling_prior_beta=feature_scaling_prior_beta,
    decoder_weight_l2=decoder_weight_l2,
    decoder_bias_multiplier=decoder_bias_multiplier,
    init_decoder_bias=init_decoder_bias,
    bg_init_gene_fraction=bg_init_gene_fraction,
    residual_library_encoder=residual_library_encoder,
    library_obs_w_prior_rate=library_obs_w_prior_rate,
    compute_pearson=compute_pearson,
    additive_bg_prior_alpha=additive_bg_prior_alpha,
    additive_bg_prior_beta=additive_bg_prior_beta,
    z_sparsity_prior=z_sparsity_prior,
    n_active_latent_per_cell=n_active_latent_per_cell,
    decoder_hidden_l1=decoder_hidden_l1,
    hidden_activation_sparsity=hidden_activation_sparsity,
    n_active_hidden_per_cell=n_active_hidden_per_cell,
    use_kl_z=use_kl_z,
    horseshoe_latent_z_prior_type=horseshoe_latent_z_prior_type,
    horseshoe_latent_z_encoder_fraction=horseshoe_latent_z_encoder_fraction,
    horseshoe_posterior_init_loc=horseshoe_posterior_init_loc,
    horseshoe_posterior_init_scale=horseshoe_posterior_init_scale,
    var_init_scale=var_init_scale,
    use_softplus_var_activation=use_softplus_var_activation,
    use_ard_z_sigma_scale=use_ard_z_sigma_scale,
)

In [ ]:
if not skip_training:
    import time
    from scvi.train import SaveCheckpoint

    checkpoint_cb = SaveCheckpoint(
        dirpath=f"{results_folder}/checkpoints",
        every_n_epochs=200,
        save_top_k=-1,
        filename="epoch-{epoch}",
    )

    _es_kwargs = {}
    if early_stopping_min_delta_per_feature is not None:
        _es_kwargs["early_stopping_min_delta_per_feature"] = early_stopping_min_delta_per_feature

    _ds_kwargs = {"num_workers": 7}
    if stratify_validation_key is not None:
        from sklearn.model_selection import train_test_split

        keys = stratify_validation_key.split("+")
        strat_labels = adata.obs[keys[0]].astype(str)
        for k in keys[1:]:
            strat_labels = strat_labels + "_" + adata.obs[k].astype(str)
        counts = strat_labels.value_counts()
        rare = counts[counts < 2].index
        if len(rare) > 0:
            strat_labels = strat_labels.replace(rare, "_rare_")
            print(f"Merged {len(rare)} rare strata into '_rare_'")
        all_idx = np.arange(adata.n_obs)
        train_idx, val_idx = train_test_split(all_idx, test_size=0.1, stratify=strat_labels.values, random_state=42)
        _ds_kwargs["external_indexing"] = [train_idx, val_idx]
        print(f"Stratified split by {stratify_validation_key}: {len(train_idx)} train, {len(val_idx)} val")

    t0 = time.time()
    model.train(
        check_val_every_n_epoch=1,
        train_size=0.9,
        max_epochs=max_epochs,
        batch_size=batch_size,
        early_stopping=True,
        early_stopping_patience=early_stopping_patience,
        early_stopping_monitor="elbo_validation",
        enable_checkpointing=True,
        callbacks=[checkpoint_cb],
        logger=wandb_loggers,
        datasplitter_kwargs=_ds_kwargs,
        plan_kwargs={"n_epochs_kl_warmup": n_epochs_kl_warmup},
        **_es_kwargs,
    )
    elapsed = time.time() - t0
    n_epochs = len(model.history_["elbo_train"])
    print(f"Training: {elapsed / 60:.1f} min, {n_epochs} epochs, {elapsed / n_epochs:.2f} s/epoch")
else:
    print("skip_training=True, skipping training")

In [ ]:
model.compute_latent_umap(adata)
print("Computed joint + per-modality latent UMAPs")

## Results

In [ ]:
# Training diagnostics
import matplotlib.pyplot as plt

_history = getattr(model, "history_", None)
_hist = _history.get("elbo_train") if isinstance(_history, dict) else None
n_epochs = len(_hist) if _hist is not None else 0
try:
    if n_epochs > 0:
        _skip = min(80, max(5, n_epochs // 2)) if n_epochs > 10 else 5
        fig = model.plot_training_diagnostics(skip_epochs=_skip)
        if not skip_training:
            log_figure_to_wandb("training_diagnostics", fig)
        plt.show()
    else:
        print("No training history available (skip_training or loaded model)")
except (ValueError, KeyError) as e:
    print(f"plot_training_diagnostics failed: {e}")
    print(f"Available history keys: {list(_history.keys()) if isinstance(_history, dict) else 'no history_'}")

In [ ]:
ref_run_name = f"{results_folder}/model"
if not skip_training:
    model.save(ref_run_name, overwrite=True)

In [ ]:
if skip_training:
    model = regularizedvi.RegularizedMultimodalVI.load(ref_run_name, adata=mdata)
    print(f"Loaded model from {ref_run_name}")

latent = model.get_latent_representation()
adata.obsm["X_regularizedvi"] = latent
print(f"Latent representation shape: {latent.shape}")

In [ ]:
k = 50
sc.pp.neighbors(adata, use_rep="X_regularizedvi", n_neighbors=k, metric="euclidean")
sc.tl.umap(adata, min_dist=0.4, spread=1.3)

resolution = 12
while resolution > 0.5:
    sc.tl.leiden(adata, resolution=resolution, flavor="igraph")
    n_clusters = adata.obs["leiden"].nunique()
    print(f"resolution={resolution:.1f}: {n_clusters} clusters")
    if n_clusters <= 150:
        break
    resolution -= 1
print(f"Final: resolution={resolution:.1f}, {n_clusters} clusters")

In [ ]:
# Technical covariates
rcParams["figure.figsize"] = 11, 11
sc.pl.umap(
    adata,
    color=["dataset", "batch", "site", "tissue", "age_group", "condition", "sex"],
    color_map="RdPu",
    ncols=1,
    palette=sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy,
    size=1,
    vmin=0,
    vmax="p99.9",
    use_raw=False,
    legend_fontsize=8,
)

In [ ]:
# Cell annotation levels
rcParams["figure.figsize"] = 11, 11
sc.pl.umap(
    adata,
    color=["harmonized_annotation", "level_1", "level_2", "level_3"],
    color_map="RdPu",
    ncols=1,
    palette=sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy,
    size=1,
    vmin=0,
    vmax="p99.9",
    use_raw=False,
    legend_fontsize=8,
)

In [ ]:
# Per-study integration highlight: each study in black, rest in light grey
rcParams["figure.figsize"] = 11, 11
datasets = adata.obs["dataset"].unique().tolist()
for ds in sorted(datasets):
    adata.obs["_highlight"] = pd.Categorical(
        np.where(adata.obs["dataset"] == ds, ds, "other"),
        categories=[ds, "other"],
    )
    sc.pl.umap(
        adata,
        color="_highlight",
        groups=[ds],
        na_color="lightgrey",
        palette={ds: "black", "other": "lightgrey"},
        size=1,
        title=f"Integration: {ds}",
    )
del adata.obs["_highlight"]

In [ ]:
# Per-study annotation plots: highlight one study's annotations, grey out the rest
rcParams["figure.figsize"] = 11, 11
annotation_levels = ["level_1", "level_2", "level_3", "harmonized_annotation"]
datasets = sorted(adata.obs["dataset"].unique().tolist())

for ds in datasets:
    mask = adata.obs["dataset"] == ds
    for level in annotation_levels:
        col_name = f"_{ds}_{level}"
        adata.obs[col_name] = pd.Categorical(np.where(mask, adata.obs[level].astype(str), np.nan))
        sc.pl.umap(
            adata,
            color=col_name,
            na_color="lightgrey",
            palette=sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy,
            size=1,
            vmax="p99.9",
            use_raw=False,
            legend_fontsize=8,
            title=f"{ds} — {level}",
        )
        del adata.obs[col_name]

In [ ]:
# QC metrics on UMAP
rcParams["figure.figsize"] = 11, 11
qc_metrics = ["total_counts", "n_genes", "mt_frac", "doublet_score", "total_fragments"]
qc_present = [m for m in qc_metrics if m in adata.obs.columns]
sc.pl.umap(
    adata,
    color=qc_present,
    color_map="RdPu",
    ncols=3,
    size=1,
    vmax="p99.9",
)

In [ ]:
# Compute CPM-like normalized layer for marker gene plotting
import scipy.sparse

norm = adata.layers["counts"].astype(np.float32).copy()
if scipy.sparse.issparse(norm):
    row_sums = np.array(norm.sum(axis=1)).ravel()
    norm = norm.multiply(1.0 / row_sums[:, None]).tocsr()
else:
    row_sums = norm.sum(axis=1, keepdims=True)
    norm = norm / row_sums
norm = norm * 10000
adata.layers["norm_counts"] = norm
print(f"Added norm_counts layer: {norm.shape}")

In [ ]:
# Marker gene UMAPs — saved to results_folder/plots/ instead of embedding
import os
import matplotlib.pyplot as plt

plots_dir = os.path.join(results_folder, "plots", "marker_genes")
os.makedirs(plots_dir, exist_ok=True)

marker_df = pd.read_csv(marker_genes_csv)
categories = marker_df.groupby("category", sort=False)["gene"].apply(lambda x: list(dict.fromkeys(x)))

rcParams["figure.figsize"] = 11, 11
for category, gene_list in categories.items():
    present = [g for g in gene_list if g in adata.var["SYMBOL"].values]
    if not present:
        print(f"  {category}: no genes found in data, skipping")
        continue
    print(f"\n{category} ({len(present)}/{len(gene_list)} genes)")
    var_names = [adata.var_names[adata.var["SYMBOL"] == g][0] for g in present]
    sc.pl.umap(
        adata,
        color=var_names,
        layer="norm_counts",
        color_map="RdPu",
        ncols=5,
        size=1,
        vmax="p99.9",
        use_raw=False,
        title=present,
        show=False,
    )
    fig = plt.gcf()
    safe_name = category.replace("/", "_").replace(" ", "_")
    fig.savefig(os.path.join(plots_dir, f"{safe_name}.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved to {plots_dir}/{safe_name}.png")

In [ ]:
# Leiden clusters
rcParams["figure.figsize"] = 11, 11
sc.pl.umap(
    adata,
    color="leiden",
    color_map="RdPu",
    ncols=1,
    palette=sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy,
    size=1,
    legend_fontsize=8,
)

In [ ]:
# Marker gene dotplots
from regularizedvi.plt import plot_marker_dotplots

dotplot_dir = os.path.join(results_folder, "plots", "dotplots")

# Single leiden dotplot (all datasets combined)
print("=== Dotplot: leiden clusters (all datasets) ===")
plot_marker_dotplots(
    adata,
    marker_genes_csv,
    groupby="leiden",
    layer="norm_counts",
    save_dir=dotplot_dir,
    per_dataset=False,
)

# Per-dataset harmonised annotation dotplots
print("\n=== Dotplots: harmonized annotations (per dataset) ===")
plot_marker_dotplots(
    adata,
    marker_genes_csv,
    groupby="harmonized_annotation",
    layer="norm_counts",
    save_dir=dotplot_dir,
    per_dataset=True,
)

In [ ]:
output_dir = f"{ref_run_name}/outputs/"
os.makedirs(output_dir, exist_ok=True)

X_regularizedvi = pd.DataFrame(
    adata.obsm["X_regularizedvi"],
    index=adata.obs_names,
    columns=range(adata.obsm["X_regularizedvi"].shape[1]),
)
X_regularizedvi.to_csv(f"{output_dir}/X_regularizedvi.csv")

X_umap = pd.DataFrame(
    adata.obsm["X_umap"],
    index=adata.obs_names,
    columns=range(2),
)
X_umap.to_csv(f"{output_dir}/X_umap_k{k}.csv")

adata.obs[["leiden"]].to_csv(f"{output_dir}/leiden_k{k}.csv")

scipy.sparse.save_npz(f"{output_dir}/distances_euclidean_k{k}.npz", adata.obsp["distances"], compressed=True)
scipy.sparse.save_npz(f"{output_dir}/connectivities_euclidean_k{k}.npz", adata.obsp["connectivities"], compressed=True)

print(f"Outputs saved to {output_dir}")

In [ ]:
# Integration metrics (after results save — wrapped in try/except for safety)
try:
    from regularizedvi.plt import compute_integration_metrics

    metrics_df = compute_integration_metrics(
        adata,
        latent_key="X_regularizedvi",
        label_key="level_1",
        batch_key="dataset",
        leiden_key="leiden",
        subsample_n=50000,
        random_state=42,
    )
    metrics_path = os.path.join(results_folder, "integration_metrics.csv")
    metrics_df.to_csv(metrics_path, index=False)
    print(f"\nSaved integration metrics to {metrics_path}")
except Exception as e:  # noqa: BLE001
    print(f"Integration metrics failed: {e}")
    import traceback

    traceback.print_exc()

In [ ]:
finish_wandb()